# Add `IntendedFor` to Fieldmap JSON Sidecars

After heudiconv converts your DICOMs to BIDS, each fieldmap scan gets a `.json` sidecar file. However, heudiconv does not know which functional runs each fieldmap is meant to correct — you need to add that information manually using the `IntendedFor` field.

**Why this matters:** fMRIPrep uses `IntendedFor` to decide which fieldmap to apply for susceptibility distortion correction (SDC) on each BOLD run. If `IntendedFor` is missing, fMRIPrep will skip SDC entirely for those runs.

This notebook loops over a list of subjects and updates each fieldmap JSON in-place.

---

### What is `IntendedFor`?

It is a BIDS-required field in fieldmap JSON files that lists the functional NIfTI files the fieldmap should be applied to:

```json
{
  "PhaseEncodingDirection": "j-",
  "IntendedFor": [
      "func/sub-001_task-image_run-1_bold.nii.gz",
      "func/sub-001_task-image_run-2_bold.nii.gz"
  ]
}
```

#### History
- 6/12/20 dcosme — modified existing code for this purpose
- Refactored for CNLab pipeline documentation

## 1. Imports

In [ ]:
import json
import os
import re

## 2. Set Paths and Subject List

In [ ]:
# Path to your BIDS directory
bids_dir = '/data00/projects/your_project/data/bids_data/'

# Option A: manually specify subjects
subs = ['sub-001', 'sub-002']

# Option B: auto-detect all subject directories in the BIDS folder
# subs = sorted([d for d in os.listdir(bids_dir) if re.match(r'sub-.*', d)])

print(f"BIDS directory : {bids_dir}")
print(f"Subjects ({len(subs)}) : {subs}")

## 3. Update File Permissions (If Needed)

If heudiconv wrote the JSON files with restricted permissions, you may need to make them writable before editing. Run the cell below if you encounter permission errors.

In [ ]:
# Uncomment if you need to update file permissions before editing JSONs
# !chmod 777 -R {bids_dir}/sub-*

## 4. Define `add_intendedfor` Function

This function reads a fieldmap JSON file, adds or updates the `IntendedFor` field with the list of functional NIfTI paths, and writes the file back in-place.

In [ ]:
def add_intendedfor(subj_dir, fmap_files, func_files):
    """
    Add or update the IntendedFor field in a fieldmap JSON sidecar.

    Parameters
    ----------
    subj_dir   : str  — Path to the subject's BIDS folder (e.g. bids_dir/sub-001)
    fmap_files : list — Relative paths to fieldmap JSON files (relative to subj_dir)
                        e.g. ['fmap/sub-001_acq-1_epi.json']
    func_files : list — Relative paths to functional NIfTI files to list in IntendedFor
                        e.g. ['func/sub-001_task-image_run-1_bold.nii.gz']
    """
    if not fmap_files:
        print('  No fieldmap JSON found — skipping')
        return
    if not func_files:
        print('  No matching functional files found — skipping')
        return

    json_path = os.path.join(subj_dir, fmap_files[0])

    with open(json_path, 'r') as f:
        data = json.load(f)

    data['IntendedFor'] = func_files

    with open(json_path, 'w') as f:
        json.dump(data, f, indent=4)

    print(f'  Updated: {fmap_files[0]}')
    print(f'  IntendedFor: {func_files}')

## 5. Map Fieldmaps to Functional Runs

Edit the matching logic in the loop below to reflect your study's acquisition structure.

**How it works:**
- For each subject, find the fieldmap JSON(s) matching a particular acquisition (e.g. `acq-1`) and the functional NIfTI(s) for the corresponding task run
- Call `add_intendedfor()` to write that mapping into the JSON

**Matching is done by filename substring.** Adjust the filter strings to match your BIDS filenames.

> **Geoscan / bbprime reference:** The bbprime project had fieldmaps `acq-1` through `acq-4` corresponding to a `read` task run and three `share` task runs respectively. Adapt the example below for your own task structure.

In [ ]:
for s in subs:
    subj_dir = os.path.join(bids_dir, s)
    fmap_dir = os.path.join(subj_dir, 'fmap')
    func_dir = os.path.join(subj_dir, 'func')

    print(f'\n{s}')
    print('─' * 40)

    # ── EDIT THIS SECTION for your study ──────────────────────────────────────
    # For each fieldmap acquisition, define:
    #   fmap  : the JSON sidecar(s) for that fieldmap (match by acquisition label or filename substring)
    #   func  : the functional NIfTI(s) that fieldmap should correct (match by task/run name)
    #
    # Example below: bbprime project with 4 fieldmap acquisitions

    # acq-1 fieldmap → read task
    print('\n  acq-1 → read task')
    fmap = [os.path.join('fmap', f) for f in os.listdir(fmap_dir)
            if 'acq-1' in f and f.endswith('.json')]
    func = [os.path.join('func', f) for f in os.listdir(func_dir)
            if 'read' in f and f.endswith('.nii.gz')]
    add_intendedfor(subj_dir, fmap, func)

    # acq-2 fieldmap → share task run 1
    print('\n  acq-2 → share task run-1')
    fmap = [os.path.join('fmap', f) for f in os.listdir(fmap_dir)
            if 'acq-2' in f and f.endswith('.json')]
    func = [os.path.join('func', f) for f in os.listdir(func_dir)
            if 'share_run-1' in f and f.endswith('.nii.gz')]
    add_intendedfor(subj_dir, fmap, func)

    # acq-3 fieldmap → share task run 2
    print('\n  acq-3 → share task run-2')
    fmap = [os.path.join('fmap', f) for f in os.listdir(fmap_dir)
            if 'acq-3' in f and f.endswith('.json')]
    func = [os.path.join('func', f) for f in os.listdir(func_dir)
            if 'share_run-2' in f and f.endswith('.nii.gz')]
    add_intendedfor(subj_dir, fmap, func)

    # acq-4 fieldmap → share task run 3 (optional — only if scan was acquired)
    fmap = [os.path.join('fmap', f) for f in os.listdir(fmap_dir)
            if 'acq-4' in f and f.endswith('.json')]
    func = [os.path.join('func', f) for f in os.listdir(func_dir)
            if 'share_run-3' in f and f.endswith('.nii.gz')]
    if fmap:
        print('\n  acq-4 → share task run-3')
        add_intendedfor(subj_dir, fmap, func)

    # ── END EDIT SECTION ──────────────────────────────────────────────────────

print('\nDone.')

## 6. Verify the Output

Check one subject's fieldmap JSON to confirm `IntendedFor` was written correctly.

In [ ]:
# Print the content of all fieldmap JSON files for the first subject
check_sub = subs[0]
fmap_dir  = os.path.join(bids_dir, check_sub, 'fmap')

json_files = sorted([f for f in os.listdir(fmap_dir) if f.endswith('.json')])

print(f'Fieldmap JSONs for {check_sub}:\n')
for fname in json_files:
    fpath = os.path.join(fmap_dir, fname)
    with open(fpath) as f:
        data = json.load(f)
    intended = data.get('IntendedFor', 'NOT SET')
    print(f'  {fname}')
    print(f'    IntendedFor: {intended}\n')

---
## Next Steps

1. Validate your BIDS dataset with the [BIDS Validator](https://bids-standard.github.io/bids-validator/) — `IntendedFor` errors should now be resolved
2. Optionally run MRIQC — see `README_2.5_mriqc.md`
3. Proceed to **fMRIPrep** — see `README_3_fmriprep.md`